In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

## Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import LabelEncoder
import wandb
import warnings
warnings.filterwarnings('ignore')

## Configuration & WandB Initialization

In [ ]:
import torch
import wandb
from kaggle_secrets import UserSecretsClient

# Fetch W&B API key securely from Kaggle secrets
user_secrets = UserSecretsClient()
wandb_api_key = user_secrets.get_secret("WanDB_API_KEY")

# Log in to W&B using the secret key
wandb.login(key=wandb_api_key)

# W&B Init (APNA ROLL NO DAAL)
wandb.init(
    project="DL-23f1000967-notebook-t22026",
    name="EDA_tfidf_baseline",
    config={
        "method": "EDA + TF-IDF + Cosine Similarity",
        "description": "Baseline model using TF-IDF embeddings"
    }
)

# # Access config and device
# config = wandb.config
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# print(f"Using device: {device}")


## Data Loading

In [ ]:
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
sample_sub = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv')

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")
print(f"\nTrain columns: {train.columns.tolist()}")
print(f"\nSample train row:")
print(train.iloc[0])

## Exploratory Data Analysis

In [ ]:
# Answer distribution
answer_counts = train['answer'].value_counts().sort_index()
print("\nAnswer Distribution:")
print(answer_counts)

In [ ]:
# Plot answer distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

In [ ]:
# Bar plot
axes[0].bar(answer_counts.index, answer_counts.values, color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7'])
axes[0].set_xlabel('Answer Label')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Answers in Training Data')
axes[0].set_xticks(['A', 'B', 'C', 'D', 'E'])

In [ ]:
# Pie chart
axes[1].pie(answer_counts.values, labels=answer_counts.index, autopct='%1.1f%%', 
           colors=['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7'])
axes[1].set_title('Answer Distribution (Percentage)')

plt.tight_layout()
plt.savefig('answer_distribution.png', dpi=150, bbox_inches='tight')
wandb.log({"answer_distribution": wandb.Image('answer_distribution.png')})
plt.show()

In [ ]:
# Text length analysis
train['prompt_length'] = train['prompt'].str.len()
train['prompt_words'] = train['prompt'].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(train['prompt_length'], bins=50, color='#4ECDC4', edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Prompt Length (characters)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Prompt Length')
axes[0].axvline(train['prompt_length'].mean(), color='red', linestyle='--', label=f'Mean: {train["prompt_length"].mean():.0f}')
axes[0].legend()

axes[1].hist(train['prompt_words'], bins=50, color='#45B7D1', edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Prompt Length (words)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Prompt Word Count')
axes[1].axvline(train['prompt_words'].mean(), color='red', linestyle='--', label=f'Mean: {train["prompt_words"].mean():.0f}')
axes[1].legend()

plt.tight_layout()
plt.savefig('prompt_length_analysis.png', dpi=150, bbox_inches='tight')
wandb.log({"prompt_length_analysis": wandb.Image('prompt_length_analysis.png')})
plt.show()

In [ ]:
# Option length analysis
for opt in ['A', 'B', 'C', 'D', 'E']:
    train[f'{opt}_length'] = train[opt].str.len()

option_lengths = [train[f'{opt}_length'].mean() for opt in ['A', 'B', 'C', 'D', 'E']]

plt.figure(figsize=(10, 5))
plt.bar(['A', 'B', 'C', 'D', 'E'], option_lengths, color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7'])
plt.xlabel('Option Label')
plt.ylabel('Average Length (characters)')
plt.title('Average Option Length by Label')
plt.savefig('option_length_analysis.png', dpi=150, bbox_inches='tight')
wandb.log({"option_length_analysis": wandb.Image('option_length_analysis.png')})
plt.show()

In [ ]:
# Log EDA stats to W&B
wandb.log({
    "train_size": len(train),
    "test_size": len(test),
    "avg_prompt_length": train['prompt_length'].mean(),
    "avg_prompt_words": train['prompt_words'].mean(),
    "min_prompt_length": train['prompt_length'].min(),
    "max_prompt_length": train['prompt_length'].max()
})

## Data Preprocessing



In [ ]:
import re

def clean_text(text):
    """Clean and preprocess text data"""
    if pd.isna(text):
        return ""
    # Convert to lowercase
    text = str(text).lower()
    
    # Remove special characters but keep spaces
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [ ]:
# Apply cleaning
for col in ['prompt', 'A', 'B', 'C', 'D', 'E']:
    train[f'{col}_clean'] = train[col].apply(clean_text)
    test[f'{col}_clean'] = test[col].apply(clean_text)

print("Text cleaning completed!")
print(f"\nSample cleaned prompt: {train['prompt_clean'].iloc[0][:200]}...")

## TF-IDF Vectorization

In [ ]:
# Combine prompt and each option for TF-IDF
def create_combined_text(row, option):
    """Combine prompt with option for TF-IDF"""
    return f"{row['prompt_clean']} {row[f'{option}_clean']}"

In [ ]:
# Create TF-IDF vectorizer
# from sklearn.feature_extraction.text import (
#     TfidfVectorizer,
#     ENGLISH_STOP_WORDS
# )
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    stop_words='english'
)

In [ ]:
# Fit on all text data
all_texts = []
for opt in ['A', 'B', 'C', 'D', 'E']:
    all_texts.extend(train[f'{opt}_clean'].tolist())
    all_texts.extend(test[f'{opt}_clean'].tolist())
all_texts.extend(train['prompt_clean'].tolist())
all_texts.extend(test['prompt_clean'].tolist())

tfidf.fit(all_texts)
print(f"TF-IDF vocabulary size: {len(tfidf.vocabulary_)}")

## Baseline Model

In [ ]:
def get_tfidf_predictions(df, tfidf, is_train=False):
    """
    Get predictions using TF-IDF cosine similarity between prompt and each option
    Returns top 3 predictions in ranked order
    """
    predictions = []
    
    for idx, row in df.iterrows():
        # Get prompt embedding
        prompt_vec = tfidf.transform([row['prompt_clean']])
        
        # Calculate similarity with each option
        similarities = {}
        for opt in ['A', 'B', 'C', 'D', 'E']:
            opt_vec = tfidf.transform([row[f'{opt}_clean']])
            sim = cosine_similarity(prompt_vec, opt_vec)[0][0]
            similarities[opt] = sim
        
        # Sort by similarity and get top 3
        sorted_options = sorted(similarities.keys(), key=lambda x: similarities[x], reverse=True)
        top_3 = ' '.join(sorted_options[:3])
        predictions.append(top_3)
    
    return predictions

In [ ]:
# Get predictions for train (for validation)
train_predictions = get_tfidf_predictions(train, tfidf, is_train=True)

In [ ]:
# Get predictions for test
test_predictions = get_tfidf_predictions(test, tfidf, is_train=False)

print(f"\nSample predictions:")
for i in range(5):
    print(f"Q{i+1}: {test_predictions[i]}")

## MAP@3 Evaluation

In [ ]:
def map_at_3(actual, predicted):
    """
    Calculate Mean Average Precision at 3 (MAP@3)
    
    Args:
        actual: List of actual answer labels
        predicted: List of predicted answer strings (space-separated top 3)
    
    Returns:
        MAP@3 score
    """
    def ap_at_3(a, p):
        """Average Precision at 3 for single prediction"""
        p = p.split()
        score = 0.0
        num_hits = 0
        
        for i, pred in enumerate(p[:3]):
            if pred == a and pred not in p[:i]:
                num_hits += 1
                score += num_hits / (i + 1)
        
        return score
    
    return np.mean([ap_at_3(a, p) for a, p in zip(actual, predicted)])

In [ ]:
# Calculate train MAP@3
train_map3 = map_at_3(train['answer'].tolist(), train_predictions)
print(f"TRAIN MAP@3 (TF-IDF Baseline): {train_map3:.4f}")

In [ ]:
# Log to W&B
wandb.log({
    "train_map3": train_map3,
    "model": "TF-IDF Cosine Similarity",
    "vocab_size": len(tfidf.vocabulary_)
})

## Scratch Model

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
import wandb
from kaggle_secrets import UserSecretsClient

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(seed=42)

# WandB already initialized above; update the run config for this model
wandb.config.update({
        "model_type": "FROM_scratch_OPTION_SCORER_PYTORCH",
        "feature_extractor": "TFIDF_word+char+SVD",
        "word_tfidf_features": 4000,
        "char_tfidf_features": 3000,
        "svd_components": 150,
        "hidden_dim": 256,
        "dropout": 0.0,
        "weight_decay": 0.0001,
        "learning_rate": 0.001,
        "batch_size": 64,
        "epochs": 250,
        "n_folds": 5
    }, allow_val_change=True)

config = wandb.config
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
OPTS = ['A', 'B', 'C', 'D', 'E']
print(f"Train Dataset Shape: {train_df.shape}")
print(f"Test Dataset Shape:  {test_df.shape}\n")

In [ ]:
def build_corpus(df):
    corpus = df['prompt'].fillna("").tolist()
    for c in OPTS:
        corpus += df[c].fillna("").tolist()
    return corpus

full_corpus = build_corpus(train_df) + build_corpus(test_df)

word_vec = TfidfVectorizer(stop_words='english', max_features=config.word_tfidf_features,
                            sublinear_tf=True, ngram_range=(1, 2), min_df=2)
word_vec.fit(full_corpus)

char_vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5),
                            max_features=config.char_tfidf_features, sublinear_tf=True, min_df=2)
char_vec.fit(full_corpus)

svd = TruncatedSVD(n_components=config.svd_components, random_state=42)
svd.fit(word_vec.transform(full_corpus))

In [ ]:
def jaccard(a, b):
    sa, sb = set(a.lower().split()), set(b.lower().split())
    if not sa or not sb:
        return 0.0
    return len(sa & sb) / len(sa | sb)

In [ ]:
def build_features(df):
    n = len(df)
    feats = np.zeros((n, 5, 8), dtype=np.float32)

    prompts = df['prompt'].fillna("").tolist()
    opt_texts = {c: df[c].fillna("").tolist() for c in OPTS}

    p_word = word_vec.transform(prompts)
    p_char = char_vec.transform(prompts)
    p_svd = svd.transform(p_word)

    opt_word = {c: word_vec.transform(opt_texts[c]) for c in OPTS}
    opt_char = {c: char_vec.transform(opt_texts[c]) for c in OPTS}
    opt_svd = {c: svd.transform(opt_word[c]) for c in OPTS}

    for i in range(n):
        p_len = max(len(prompts[i]), 1)
        p_words = max(len(prompts[i].split()), 1)
        opt_lens = [len(opt_texts[c][i]) for c in OPTS]
        opt_word_counts = [len(opt_texts[c][i].split()) for c in OPTS]

        word_cos = np.array([cosine_similarity(p_word[i], opt_word[c][i])[0, 0] for c in OPTS])
        char_cos = [cosine_similarity(p_char[i], opt_char[c][i])[0, 0] for c in OPTS]
        svd_cos = [cosine_similarity(p_svd[i:i+1], opt_svd[c][i:i+1])[0, 0] for c in OPTS]
        jac = [jaccard(prompts[i], opt_texts[c][i]) for c in OPTS]
        len_ratio = [l / p_len for l in opt_lens]
        wc_ratio = [w / p_words for w in opt_word_counts]

        opt_vecs = [opt_word[c][i] for c in OPTS]
        centrality = [np.mean([cosine_similarity(opt_vecs[a], opt_vecs[b])[0, 0]
                                for b in range(5) if b != a]) for a in range(5)]

        word_cos_z = (word_cos - word_cos.mean()) / (word_cos.std() + 1e-6)

        for k in range(5):
            feats[i, k] = [word_cos[k], char_cos[k], svd_cos[k], jac[k],
                           len_ratio[k], wc_ratio[k], centrality[k], word_cos_z[k]]
    return feats

In [ ]:
X_train_raw = build_features(train_df)
X_test_raw = build_features(test_df)

encoder = LabelEncoder()
y_train_raw = encoder.fit_transform(train_df['answer'])

print(f"Engineered Tensor (Train): {X_train_raw.shape}")
print(f"Engineered Tensor (Test):  {X_test_raw.shape}")

In [ ]:
class OptionScorer(nn.Module):
    def __init__(self, in_features, hidden_dim, dropout):
        super(OptionScorer, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, x):
        B, K, F = x.shape
        return self.net(x.reshape(B * K, F)).reshape(B, K)

def map_at_3(probs, y_true):
    top3 = np.argsort(-probs, axis=1)[:, :3]
    score = 0.0
    for i in range(len(y_true)):
        ranks = list(top3[i])
        if y_true[i] in ranks:
            score += 1.0 / (ranks.index(y_true[i]) + 1)
    return score / len(y_true)

In [ ]:
num_feats = X_train_raw.shape[2]
skf = StratifiedKFold(n_splits=config.n_folds, shuffle=True, random_state=42)

oof_probs = np.zeros((len(X_train_raw), 5))
test_probs_sum = np.zeros((len(X_test_raw), 5))
fold_scores = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train_raw, y_train_raw)):
    Xtr, Xva = X_train_raw[tr_idx], X_train_raw[va_idx]
    ytr, yva = y_train_raw[tr_idx], y_train_raw[va_idx]

    mean = Xtr.reshape(-1, num_feats).mean(0)
    std = Xtr.reshape(-1, num_feats).std(0) + 1e-6
    Xtr_n, Xva_n = (Xtr - mean) / std, (Xva - mean) / std
    Xte_n = (X_test_raw - mean) / std

    Xtr_t = torch.tensor(Xtr_n, dtype=torch.float32).to(device)
    ytr_t = torch.tensor(ytr, dtype=torch.long).to(device)
    Xva_t = torch.tensor(Xva_n, dtype=torch.float32).to(device)
    Xte_t = torch.tensor(Xte_n, dtype=torch.float32).to(device)

    model = OptionScorer(num_feats, config.hidden_dim, config.dropout).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
    criterion = nn.CrossEntropyLoss()
    loader = DataLoader(TensorDataset(Xtr_t, ytr_t), batch_size=config.batch_size, shuffle=True)

    for epoch in range(1, config.epochs + 1):
        model.train()
        for bx, by in loader:
            optimizer.zero_grad()
            loss = criterion(model(bx), by)
            loss.backward()
            optimizer.step()

        if epoch % 20 == 0 or epoch == config.epochs:
            model.eval()
            with torch.no_grad():
                va_probs = torch.softmax(model(Xva_t), dim=1).cpu().numpy()
            wandb.log({"fold": fold, "epoch": epoch, "val_map3": map_at_3(va_probs, yva)})

    model.eval()
    with torch.no_grad():
        va_probs = torch.softmax(model(Xva_t), dim=1).cpu().numpy()
        te_probs = torch.softmax(model(Xte_t), dim=1).cpu().numpy()

    fold_map3 = map_at_3(va_probs, yva)
    oof_probs[va_idx] = va_probs
    test_probs_sum += te_probs
    fold_scores.append(fold_map3)
    print(f"Fold {fold} | val MAP@3: {fold_map3:.4f}")

In [ ]:
oof_map3 = map_at_3(oof_probs, y_train_raw)
wandb.log({"mean_fold_map3": np.mean(fold_scores), "oof_map3": oof_map3})
wandb.finish()
print(f"Mean fold MAP@3: {np.mean(fold_scores):.4f}")
print(f"Overall OOF MAP@3: {oof_map3:.4f}")

## inference for scratch model

In [ ]:
test_probs = test_probs_sum / config.n_folds
classes = np.array(OPTS)
map_predictions = []

for row_probs in test_probs:
    top_3_indices = np.argsort(row_probs)[::-1][:3]
    top_3_labels = classes[top_3_indices]
    map_predictions.append(" ".join(top_3_labels))

submission_df = pd.DataFrame({
    'id': test_df['id'],
    'prediction': map_predictions
})
display(submission_df.head(5))

## Pretrained model(DeBERTa-v3-base)

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForMultipleChoice, get_linear_schedule_with_warmup
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
import wandb
from kaggle_secrets import UserSecretsClient

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(seed=42)

# WandB already initialized above; update the run config for this model
wandb.config.update({
        "model_name": "microsoft/deberta-v3-base",
        "max_length": 256,
        "batch_size": 8,
        "grad_accum_steps": 2,
        "epochs": 3,
        "learning_rate": 2e-5,
        "weight_decay": 0.01,
        "warmup_ratio": 0.1,
        "n_folds": 5
    }, allow_val_change=True)

config = wandb.config
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
OPTS = ['A', 'B', 'C', 'D', 'E']

encoder = LabelEncoder()
train_df['label'] = encoder.fit_transform(train_df['answer'])

print(f"Train Dataset Shape: {train_df.shape}")
print(f"Test Dataset Shape:  {test_df.shape}\n")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(config.model_name)

class MCQDataset(Dataset):
    def __init__(self, df, tokenizer, max_length, has_labels=True):
        self.prompts = df['prompt'].fillna("").tolist()
        self.options = df[OPTS].fillna("").values.tolist()
        self.labels = df['label'].tolist() if has_labels else None
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.prompts)

    def __getitem__(self, idx):
        prompt = self.prompts[idx]
        opts = self.options[idx]
        encoded = self.tokenizer(
            [prompt] * 5, opts,
            truncation=True, max_length=self.max_length,
            padding='max_length', return_tensors='pt'
        )
        item = {k: v for k, v in encoded.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

## MAP@3 Metric

In [ ]:
def map_at_3(probs, y_true):
    top3 = np.argsort(-probs, axis=1)[:, :3]
    score = 0.0
    for i in range(len(y_true)):
        ranks = list(top3[i])
        if y_true[i] in ranks:
            score += 1.0 / (ranks.index(y_true[i]) + 1)
    return score / len(y_true)

## Traning loop for pretrained model

In [ ]:
skf = StratifiedKFold(n_splits=config.n_folds, shuffle=True, random_state=42)

oof_probs = np.zeros((len(train_df), 5))
test_probs_sum = np.zeros((len(test_df), 5))
fold_scores = []

test_dataset = MCQDataset(test_df, tokenizer, config.max_length, has_labels=False)
test_loader = DataLoader(test_dataset, batch_size=config.batch_size, shuffle=False)

In [ ]:
for fold, (tr_idx, va_idx) in enumerate(skf.split(train_df, train_df['label'])):
    train_fold = train_df.iloc[tr_idx].reset_index(drop=True)
    val_fold = train_df.iloc[va_idx].reset_index(drop=True)

    train_dataset = MCQDataset(train_fold, tokenizer, config.max_length)
    val_dataset = MCQDataset(val_fold, tokenizer, config.max_length)
    train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False)

    model = AutoModelForMultipleChoice.from_pretrained(config.model_name, torch_dtype=torch.float32).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
    total_steps = (len(train_loader) // config.grad_accum_steps) * config.epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(total_steps * config.warmup_ratio), num_training_steps=total_steps)
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))

    for epoch in range(1, config.epochs + 1):
        model.train()
        running_loss = 0.0
        optimizer.zero_grad()
        for step, batch in enumerate(train_loader):
            batch = {k: v.to(device) for k, v in batch.items()}
            with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
                outputs = model(**batch)
                loss = outputs.loss / config.grad_accum_steps
            scaler.scale(loss).backward()
            running_loss += loss.item() * config.grad_accum_steps

            if (step + 1) % config.grad_accum_steps == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad()

        epoch_loss = running_loss / len(train_loader)
        wandb.log({"fold": fold, "epoch": epoch, "train_loss": epoch_loss})
        print(f"Fold {fold} | Epoch {epoch}/{config.epochs} | train loss: {epoch_loss:.4f}")

    model.eval()
    val_probs = []
    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(device) for k, v in batch.items() if k != 'labels'}
            with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
                logits = model(**batch).logits
            val_probs.append(torch.softmax(logits, dim=1).float().cpu().numpy())
    val_probs = np.concatenate(val_probs, axis=0)
    fold_map3 = map_at_3(val_probs, val_fold['label'].values)
    oof_probs[va_idx] = val_probs
    fold_scores.append(fold_map3)
    wandb.log({"fold": fold, "val_map3": fold_map3})
    print(f"Fold {fold} | val MAP@3: {fold_map3:.4f}")

    test_probs = []
    with torch.no_grad():
        for batch in test_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
                logits = model(**batch).logits
            test_probs.append(torch.softmax(logits, dim=1).float().cpu().numpy())
    test_probs_sum += np.concatenate(test_probs, axis=0)

    del model
    torch.cuda.empty_cache()

In [ ]:
oof_map3 = map_at_3(oof_probs, train_df['label'].values)
wandb.log({"mean_fold_map3": np.mean(fold_scores), "oof_map3": oof_map3})
wandb.finish()
print(f"Mean fold MAP@3: {np.mean(fold_scores):.4f}")
print(f"Overall OOF MAP@3: {oof_map3:.4f}")

# inference for pretrained model

In [ ]:
test_probs = test_probs_sum / config.n_folds
classes = np.array(OPTS)
map_predictions = []

for row_probs in test_probs:
    top_3_indices = np.argsort(row_probs)[::-1][:3]
    top_3_labels = classes[top_3_indices]
    map_predictions.append(" ".join(top_3_labels))

submission_df = pd.DataFrame({
    'id': test_df['id'],
    'prediction': map_predictions
})
display(submission_df.head(5))

## RoBERTa model

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForMultipleChoice, get_linear_schedule_with_warmup
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
import wandb
from kaggle_secrets import UserSecretsClient

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(seed=42)

# WandB already initialized above; update the run config for this model
wandb.config.update({
        "model_name": "roberta-base",
        "max_length": 256,
        "batch_size": 8,
        "grad_accum_steps": 2,
        "epochs": 3,
        "learning_rate": 2e-5,
        "weight_decay": 0.01,
        "warmup_ratio": 0.1,
        "n_folds": 5
    }, allow_val_change=True)

config = wandb.config
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(config.model_name)

## Training loop for Roberta model

In [ ]:
skf = StratifiedKFold(n_splits=config.n_folds, shuffle=True, random_state=42)

oof_probs = np.zeros((len(train_df), 5))
test_probs_sum = np.zeros((len(test_df), 5))
fold_scores = []

test_dataset = MCQDataset(test_df, tokenizer, config.max_length, has_labels=False)
test_loader = DataLoader(test_dataset, batch_size=config.batch_size, shuffle=False)

In [ ]:
for fold, (tr_idx, va_idx) in enumerate(skf.split(train_df, train_df['label'])):
    train_fold = train_df.iloc[tr_idx].reset_index(drop=True)
    val_fold = train_df.iloc[va_idx].reset_index(drop=True)

    train_dataset = MCQDataset(train_fold, tokenizer, config.max_length)
    val_dataset = MCQDataset(val_fold, tokenizer, config.max_length)
    train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False)

    model = AutoModelForMultipleChoice.from_pretrained(config.model_name, torch_dtype=torch.float32).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
    total_steps = (len(train_loader) // config.grad_accum_steps) * config.epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(total_steps * config.warmup_ratio), num_training_steps=total_steps)
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))

    for epoch in range(1, config.epochs + 1):
        model.train()
        running_loss = 0.0
        optimizer.zero_grad()
        for step, batch in enumerate(train_loader):
            batch = {k: v.to(device) for k, v in batch.items()}
            with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
                outputs = model(**batch)
                loss = outputs.loss / config.grad_accum_steps
            scaler.scale(loss).backward()
            running_loss += loss.item() * config.grad_accum_steps

            if (step + 1) % config.grad_accum_steps == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad()

        epoch_loss = running_loss / len(train_loader)
        wandb.log({"fold": fold, "epoch": epoch, "train_loss": epoch_loss})
        print(f"Fold {fold} | Epoch {epoch}/{config.epochs} | train loss: {epoch_loss:.4f}")

    model.eval()
    val_probs = []
    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(device) for k, v in batch.items() if k != 'labels'}
            with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
                logits = model(**batch).logits
            val_probs.append(torch.softmax(logits, dim=1).float().cpu().numpy())
    val_probs = np.concatenate(val_probs, axis=0)
    fold_map3 = map_at_3(val_probs, val_fold['label'].values)
    oof_probs[va_idx] = val_probs
    fold_scores.append(fold_map3)
    wandb.log({"fold": fold, "val_map3": fold_map3})
    print(f"Fold {fold} | val MAP@3: {fold_map3:.4f}")

    test_probs = []
    with torch.no_grad():
        for batch in test_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
                logits = model(**batch).logits
            test_probs.append(torch.softmax(logits, dim=1).float().cpu().numpy())
    test_probs_sum += np.concatenate(test_probs, axis=0)

    del model
    torch.cuda.empty_cache()

In [ ]:
oof_map3 = map_at_3(oof_probs, train_df['label'].values)
wandb.log({"mean_fold_map3": np.mean(fold_scores), "oof_map3": oof_map3})
wandb.finish()
print(f"Mean fold MAP@3: {np.mean(fold_scores):.4f}")
print(f"Overall OOF MAP@3: {oof_map3:.4f}")

## Final Submission (RoBERTa Only)

In [ ]:
test_probs = test_probs_sum / config.n_folds
classes = np.array(OPTS)
map_predictions = []

for row_probs in test_probs:
    top_3_indices = np.argsort(row_probs)[::-1][:3]
    top_3_labels = classes[top_3_indices]
    map_predictions.append(" ".join(top_3_labels))

submission_df = pd.DataFrame({
    'id': test_df['id'],
    'prediction': map_predictions
})
submission_df.to_csv('submission.csv', index=False)
display(submission_df.head(5))